In [2]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
from pypdf import PdfReader, PdfWriter

def resize_mixed_pdf_to_a4(input_path, output_path):
    # 標準 A4 的邊長定義
    A4_SHORT = 595.27
    A4_LONG = 841.89
    
    reader = PdfReader(input_path)
    writer = PdfWriter()
    
    for page in reader.pages:
        # 取得原頁面尺寸
        orig_width = float(page.mediabox.width)
        orig_height = float(page.mediabox.height)
        
        # 自動判斷原頁面是直的還是橫的，並決定目標 A4 的長寬
        if orig_width > orig_height:
            # 橫向頁面
            target_width = A4_LONG
            target_height = A4_SHORT
        else:
            # 直向頁面
            target_width = A4_SHORT
            target_height = A4_LONG
            
        # 計算等比例縮放係數
        scale_x = target_width / orig_width
        scale_y = target_height / orig_height
        scale = min(scale_x, scale_y)
        
        # 1. 縮放原始頁面
        page.scale_by(scale)
        
        # 2. 建立符合該頁面方向的標準 A4 空白頁
        new_page = writer.add_blank_page(width=target_width, height=target_height)
        
        # 3. 疊加頁面並強制校正邊界
        new_page.merge_page(page)
        new_page.mediabox.left = 0
        new_page.mediabox.bottom = 0
        new_page.mediabox.right = target_width
        new_page.mediabox.top = target_height
        
    # 確保輸出目錄存在
    if os.path.dirname(output_path):
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
    with open(output_path, "wb") as f:
        writer.write(f)




In [9]:
# 執行直橫混合版函式
resize_mixed_pdf_to_a4("北科_資工.pdf", "generated/北科_資工.pdf")